In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch
from torch import  nn
import random
import os
import numpy as np
from torch.optim import Adam
import matplotlib.pyplot as plt

In [ ]:
class Actor(nn.Module):
    def __init__(self,model_path):
        super().__init__()
        self.model=AutoModelForCausalLM.from_pretrained(model_path)
    def generate(self,input_ids,**kwargs):
        attention_mask=torch.ones(input_ids.shape,dtype=torch.long,device=input_ids.device)
        outputs=self.model.generate(input_ids=input_ids,attention_mask=attention_mask,pad_token_id=kwargs.get('pad_token_id', None),max_length=kwargs.get('max_length'))
        pad_token_id=kwargs.get('pad_token_id', None)
        attention_mask = outputs.not_equal(pad_token_id).to(dtype=torch.long, device=outputs.device)
        return outputs, attention_mask
    @torch.no_grad()
    def forward(self,input_ids,attention_mask):
        logits=self.model(input_ids,attention_mask=attention_mask)['logits']
        log_probs=log_probs_from_logits(logits[:,:-1,:],input_ids[:,1:])
        log_probs = log_probs.detach().requires_grad_()  # 确保log_probs有梯度
        return log_probs
    def save_pretrained(self,output_dir):
        self.model.save_pretrained(output_dir)

In [ ]:
class RewardModel(nn.Module):
    def __init__(self,model_path,embed_size):
        super().__init__()
        self.model=AutoModelForCausalLM.from_pretrained(model_path)
        self.value_fn=nn.Linear(embed_size,1,bias=False)
    def forward(self,input_ids,attention_mask,prompt_length):
        hidden=self.model(input_ids,attention_mask=attention_mask)[0]
        values=self.value_fn(hidden).squeeze(-1)
        batch_size,seq_len=input_ids.shape[0],input_ids.shape[1]
        scores=[]
        for i in range(batch_size):
            input_id=input_ids[i]
            value=values[i]
            idxs=(input_id[prompt_length:]==0).nonzero()
            idx=idxs[0].item() + prompt_length if len(idxs) > 0 else seq_len
            scores.append(value[idx-1])
        scores=torch.stack(scores)
        return values,scores

In [ ]:
def log_probs_from_logits(logits,labels):
    probs=torch.log_softmax(logits,dim=-1)
    probs_labels = probs.gather(dim=-1, index=labels.unsqueeze(-1))
    probs_labels = probs_labels.squeeze(-1)
    return probs_labels

def advantages_and_returns(values,rewards,start_ids):
    gamma=1.0
    lam=0.95
    lastgaelam=0
    advantages_reversed=[]
    seq_len=rewards.size()[-1]
    for t in reversed(range(start_ids, seq_len)):
        nextvalues=values[:, t + 1] if t < seq_len - 1 else 0.0
        delta = rewards[:,t]+gamma * nextvalues - values[:, t]
        lastgaelam = delta + gamma * lam * lastgaelam
        advantages_reversed.append(lastgaelam)
    advantages = torch.stack(advantages_reversed[::-1], dim=1).detach()
    returns = advantages + values[:,start_ids+1:]
    return advantages, returns

In [ ]:
def make_experience(actor,critic,ref,value_model,inputs_ids,kwargs):
    actor.eval()
    critic.eval()
    with torch.no_grad():
        prompt_length=inputs_ids.shape[1]
        seq_outputs,attention_mask=actor.generate(inputs_ids,**kwargs)
        action_log_probs=actor(seq_outputs,attention_mask)
        ref_action_log_probs=ref(seq_outputs,attention_mask)
        value,_=critic(seq_outputs,attention_mask,prompt_length)
        value=value[:-1]
        _,reward_score=value_model.forward(seq_outputs,attention_mask,prompt_length=prompt_length)
        reward_clip=torch.clamp(reward_score,-5,5)
        kl=-0.02*(action_log_probs-ref_action_log_probs)
        rewards=kl
        start_ids=inputs_ids.shape[1]-1
        action_mask=attention_mask[:,1:]
        ends_ids=start_ids+action_mask[:,start_ids:].sum(1)

        batch_size=action_log_probs.shape[0]
        for j in range(batch_size):
            rewards[j,start_ids:ends_ids[j]][-1]+=reward_clip[j]
        advantages,returns=advantages_and_returns(value,rewards,start_ids)
    experience={
        "input_ids": inputs_ids, "seq_outputs": seq_outputs, "attention_mask": attention_mask,
        "action_log_probs": action_log_probs, "value": value, "reward_score": reward_score,
        "advantages": advantages, "returns": returns
    }
    return experience

In [ ]:
def actor_loss_function(logprobs, old_logprobs, advantages, actions_mask, policy_clip_eps):
    log_ratio = (logprobs - old_logprobs) * actions_mask
    ratio = torch.exp(log_ratio)
    pg_loss1 = -advantages * ratio
    pg_loss2 = -advantages * torch.clamp(ratio, 1.0 - policy_clip_eps, 1.0 + policy_clip_eps)
    pg_loss = torch.sum(torch.max(pg_loss1, pg_loss2) * actions_mask) / actions_mask.sum()
    return pg_loss

def critic_loss_function(values, old_values, returns, actions_mask, value_clip_eps):
    values_clipped = torch.clamp(values, old_values - value_clip_eps, old_values + value_clip_eps)
    vf_loss1 = (values - returns) ** 2
    vf_loss2 = (values_clipped - returns) ** 2
    vf_loss = 0.5 * torch.sum(torch.max(vf_loss1, vf_loss2) * actions_mask) / actions_mask.sum()
    return vf_loss

In [ ]:
def update_model(experience_list, actor, actor_optimizer, critic, critic_optimizer,ppo_step):
    actor.train()
    critic.train()
    for _ in range(10):
        random.shuffle(experience_list)
        for i_e, experience in enumerate(experience_list):
            ppo_step += 1
            start_ids = experience["input_ids"].size()[-1] - 1
            action_log_probs = actor(experience["seq_outputs"], experience["attention_mask"])
            action_mask = experience["attention_mask"][:, 1:]
            actor_loss = actor_loss_function(action_log_probs[:, start_ids:],
                                             experience["action_log_probs"][:, start_ids:], experience["advantages"],
                                             action_mask[:, start_ids:], 0.2)
            actor_loss.backward()

            torch.nn.utils.clip_grad_norm_(actor.parameters(),1.0)
            actor_optimizer.step()
            actor_optimizer.zero_grad()
            value, _ = critic(experience["seq_outputs"], experience["attention_mask"],
                                    experience["input_ids"].size()[-1])
            value = value[:, :-1]

            critic_loss = critic_loss_function(value[:, start_ids:], experience["value"][:, start_ids:-1],
                                               experience["returns"], action_mask[:, start_ids:], 0.2)
            critic_loss.backward()

            torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
            critic_optimizer.step()
            critic_optimizer.zero_grad()
    return ppo_step

In [ ]:
def train(eposide,timesteps,renew_per_step,ori_model, actor_model, reward_model, critic_model, tokenizer, dataset, device,query_len ,max_length,batch_size=2):
    # 根据actor模型和critic模型构建actor优化器和critic优化器
    actor_optimizer = Adam(actor_model.parameters(), lr=1e-5, eps=1e-5)
    critic_optimizer = Adam(critic_model.parameters(), lr=1e-5, eps=1e-5)

    cnt_timesteps = 0
    ppo_step = 0
    experience_list = []
    mean_reward = []
    # 训练
    for i in range(eposide):
        for timestep in range(timesteps):
            cnt_timesteps += 1
            prompt_list = random.sample(dataset,batch_size)
            input_ids = tokenizer.batch_encode_plus(prompt_list, return_tensors="pt",
                                                    max_length=max_length-query_len-3,
                                                    truncation=True, padding='max_length')["input_ids"]
            input_ids = input_ids.to(device)

            generate_kwargs = {
                "max_length": input_ids.shape[1]+10,
                "pad_token_id": tokenizer.pad_token_id,
                "eos_token_id": tokenizer.eos_token_id,
            }
            # 生成经验数据，并添加到经验池中
            experience = make_experience(actor_model, critic_model, ori_model, reward_model, input_ids,
                                         generate_kwargs)
            experience_list.append(experience)
            mean_reward.extend(experience["reward_score"].detach().cpu().numpy().tolist())
            if (cnt_timesteps % renew_per_step == 0) and (cnt_timesteps != 0):
                mr = np.mean(np.array(mean_reward))
                print("时间步",cnt_timesteps,"平均奖励",mr)
                actor_model.train()
                critic_model.train()
                ppo_step = update_model(experience_list, actor_model, actor_optimizer, critic_model,
                                        critic_optimizer, ppo_step)
                experience_list = []
                mean_reward = []
        # 模型保存
        actor_model.save_pretrained(os.path.join("output_dir", "checkpoint-{}".format(ppo_step)))
        tokenizer.save_pretrained(os.path.join("output_dir", "checkpoint-{}".format(ppo_step)))

In [ ]:
def read_values_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        values = [line.strip() for line in file.readlines()]
    return values


In [ ]:

def main():
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

    #actor模型地址
    actor_path=""
    #critic模型地址
    critic_path=""
    #分词器模型地址
    tokenizer_path=""
    #嵌入层大小
    embed_size=50272
    #迭代次数
    eposide=5
    #时间步长度
    time_steps=10
    #到特定时间步就更新模型
    renew_per_steps=5
    #生成问题大小
    query_len=200
    #最大句子长度
    max_len=200
    #训练集地址
    txt_file = './data.txt'
    datasets=read_values_from_file(txt_file)

    ori_model = Actor(actor_path)
    ori_model.to(device)
    actor_model = Actor(actor_path)
    actor_model.to(device)

    reward_model = RewardModel(critic_path,embed_size)
    reward_model.to(device)

    critic_model = RewardModel(critic_path,embed_size)
    critic_model.to(device)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, padding_side='left')
    tokenizer.eos_token_id = tokenizer.sep_token_id

    train(eposide,time_steps,renew_per_steps,ori_model, actor_model, reward_model, critic_model, tokenizer, dataset, device,query_len,max_len,batch_size=2)


In [ ]:
if __name__ == '__main__':
    main()